# Create HMRF (Health and Medical Research Fund, Hong Kong) Awards

Awards from the Hong Kong Health Bureau Research Fund Secretariat funded-project
database (rfs1.healthbureau.gov.hk), harvested via the first-party REST API
(quickSearch list + getProjectDetailByFPID). ~3,966 projects, HKD, with native
reference number (grant id), project title, abstract, approved amount, call
year, lead applicant (PI) + affiliation, and the sub-fund / health category as
scheme. The HMRF umbrella includes the predecessor funds folded into it in 2011
(RFCID, HHSRF, HCPF) — carried in `fund` as funder_scheme.

PI names are family-name-first (HK/Chinese convention) and were split in the
scraper (lead_given_name / lead_family_name). Amounts are HKD (single-country,
hardcoded). provenance `hmrf`, priority 322. F4320335055 (Path A).

Source parquet: s3://openalex-ingest/awards/hmrf/hmrf_projects.parquet
Built by scripts/local/hmrf_to_s3.py

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hmrf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hmrf/hmrf_projects.parquet`;


In [ ]:
%sql
-- Check row count (should be ~3,966)
SELECT COUNT(*) as total_projects FROM openalex.awards.hmrf_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.hmrf_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.hmrf_raw LIMIT 5;

## Step 2: Create HMRF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hmrf_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320335055  -- HMRF (Hong Kong)
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.ref_no)))) % 9000000000 as id,
        g.project_title as display_name,
        COALESCE(g.abstract_final, g.abstract_proposal) as description,
        f.funder_id,
        g.ref_no as funder_award_id,
        TRY_CAST(g.approved_amount_hkd AS DECIMAL(18,2)) as amount,
        'HKD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.fund as funder_scheme,
        'hmrf' as provenance,
        CASE WHEN TRY_CAST(g.call_year AS INT) IS NOT NULL
             THEN TRY_TO_DATE(CONCAT(g.call_year, '-01-01'), 'yyyy-MM-dd') END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.call_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.lead_family_name IS NOT NULL THEN
                struct(
                    g.lead_given_name as given_name,
                    g.lead_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.lead_affiliation as name,
                        'Hong Kong' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        CAST(NULL AS STRING) as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.hmrf_raw g
    CROSS JOIN the_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hmrf' AND priority = 322;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    322 as priority
FROM openalex.awards.hmrf_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count
SELECT COUNT(*) as total_awards FROM openalex.awards.hmrf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year
FROM openalex.awards.hmrf_awards LIMIT 10;

In [ ]:
%sql
-- Funder distribution (should all be HMRF)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.hmrf_awards GROUP BY funder.display_name ORDER BY cnt DESC;

In [ ]:
%sql
-- Scheme (sub-fund) distribution
SELECT funder_scheme, COUNT(*) as cnt,
  ROUND(SUM(amount)/1000000, 2) as funding_million_hkd
FROM openalex.awards.hmrf_awards GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.hmrf_awards WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;

In [ ]:
%sql
-- Data completeness + amount coverage (Step 6.7 fail-fast)
SELECT
  COUNT(*) as total,
  COUNT(display_name) as has_title,
  COUNT(description) as has_abstract,
  COUNT(amount) as has_amount,
  COUNT(lead_investigator) as has_pi,
  ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
  ROUND(SUM(amount)/1000000, 2) as total_funding_million_hkd
FROM openalex.awards.hmrf_awards;

In [ ]:
%sql
-- PI frequency check (Step 6.4a — top families should be a real long-tail of HK surnames)
SELECT lead_investigator.given_name as given, lead_investigator.family_name as family, COUNT(*) as n
FROM openalex.awards.hmrf_awards
GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Top institutions
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as cnt
FROM openalex.awards.hmrf_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- Verify rows landed in openalex_awards_raw
SELECT COUNT(*) as hmrf_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hmrf' AND priority = 322;